In [2]:
import yfinance as yf
import csv
import requests
import pandas as pd
import os
from utils.logger import get_logger
from datetime import datetime, timedelta
from utils.paths import DATA_DIR
from utils.config import yaml_read
import re

In [17]:
class YahooFinanceClient:

    def __init__(self):
        self.logger = get_logger(__name__)
        self.yfinance_data = None

    def save_to_csv(self, data, ticker_name, startDate: str, endDate : str = ""):
        """
        Save a pandas DataFrame to a CSV file.

        Parameters:
            data (pd.DataFrame): The DataFrame to be saved.
            filename (str): The name of the output CSV file.

        Returns:
            None
        """
        filename = self.csv_file_name_generator(ticker_name, startDate, endDate)
        file_path = DATA_DIR / filename

        if not data.empty:
            # Resolve multi-index columns 
            if isinstance(data.columns, pd.MultiIndex):
                data.columns = data.columns.get_level_values(0)

            data["Date"] = [startDate] * len(data)
            data["Asset"] = self.normalize_symbol(ticker_name)

            #Change column order -> set "Date" forward
            cols = ["Date", "Asset"] + [col for col in data.columns if col not in ["Date", "Asset"]]
            data = data[cols]
    
            data.to_csv(file_path, index=False)
            self.logger.debug(file_path / " is safed as csv")
     
    def normalize_symbol(self, symbole: str) -> str:
        # Remove all Whitespaces
        if symbole is None:
            return ""
        return re.sub(r"\s+", "", symbole)

    def extract_meta(self, info: dict, symbole: str) -> dict:
        return {
            "symbol": symbole,
            "name":  info.get("longName") or info.get("shortName"),
            "country": (
                info.get("country") or
                info.get("region") or
                "Global"
            ),
            "industry": (
                info.get("industry") or
                info.get("sector") or
                info.get("quoteType") or
                "Unknown"
            )
        }    

    def  fetch_finance_data(self, ticker_symbol = "DAX", date="2025-10-04", endDate = ""):

        #ticker_symbol = self.ticker_dic[ticker_name]
        date_obj = datetime.strptime(date, "%Y-%m-%d")

        try: 
            if endDate !=  "":
                self.yfinance_data = yf.download(ticker_symbol, start=date, end= endDate, timeout=5)
            else:
                self.yfinance_data = yf.download(ticker_symbol, start=date, end= date_obj + timedelta(days=1), timeout=5)
            
            #<symbol>_<YYYY-MM-DD>.csv
            if self.yfinance_data.empty:
                # https://query1.finance.yahoo.com
                self.logger.error(f"{ticker_symbol} yFinace data are empty ")
            else:
                self.save_to_csv(self.yfinance_data, ticker_symbol, date, endDate)
        

        # Handles slow or unreachable servers
        except requests.exceptions.Timeout:
           self.logger.error("Request timed out.")

        # Catches DNS failures
        except requests.exceptions.ConnectionError:
            self.logger.error("Failed to connect to the API.")

        # Cates HTTP errors like 404, 500...
        except requests.exceptions.HTTPError as err:
            self.logger.error(f"HTTP error occurred: {err}")

        # A Catch for everything else
        except requests.exceptions.RequestException as err:
            self.logger.error(f"Unexpected error: {err}")

    def fetch_company_info(self, symbole: str):
        """
        returns a Company with all Meta-Infos needed for dim_country: table 
        Keys: symbole, 
        """
        index_mapping = yaml_read("index_mapping.yaml")
        symbole = self.normalize_symbol(symbole)
        try:
            company = yf.Ticker(index_mapping["indices"][symbole])
            return self.extract_meta(company.info, symbole)
        except:
            self.logger.error(f"Unknown Key, cant insert {symbole}")
        return None
    
    def csv_file_name_generator(self, symbole: str, startDate: str, endDate: str = ""):
        if endDate == "":
            return symbole + "_" + startDate + ".csv"
        else:
            return symbole + "_" + startDate + "_to_" + endDate + ".csv"
        
        

In [ ]:
fetcher = YahooFinanceClient()
fetcher.fetch_finance_data("DAX", "2025-09-01", "2025-09-30")


C:\Users\chris\AppData\Local\Temp\ipykernel_10664\1763694231.py:66: FutureWarning: YF.download() has changed argument auto_adjust default to True
  self.yfinance_data = yf.download(ticker_symbol, start=date, end= endDate, timeout=5)
[*********************100%***********************]  1 of 1 completed
2025-10-03 15:33:19,325 - __main__ - DEBUG - C:\Users\chris\Desktop\vs Code\Python\FinDataWarehouse\data\DAX_2025-09-01_to_2025-09-30.csv\ is safed as csv
